In [45]:
import os

In [46]:
os.environ["MLFLOW_TRACKING_URI"] = "https://dagshub.com/jkbanjarey/MLOPs_Projects.mlflow"
os.environ["MLFLOW_TRACKING_USERNAME"]="jkbanjarey"
os.environ["MLFLOW_TRACKING_PASSWORD"]="0ad1f49efbdf3b7af28e4414ada709639f3b0504"



In [47]:
%pwd

'd:\\Github Projects\\Pratice'

In [ ]:
os.chdir("../")

In [48]:
%pwd

'd:\\Github Projects\\Pratice'

Entity

In [49]:
from dataclasses import dataclass
from pathlib import Path

@dataclass
class ModelEvaluationConfig:
    root_dir: Path
    test_data_path: Path
    model_path: Path
    all_params: dict
    metrics_file_name: Path
    target_column: str
    mlflow_uri: str

configuration manager

In [50]:
from src.datascience.constants import *
from src.datascience.utils.common import read_yaml, create_directories, save_json

In [51]:
import sys

class ConfigurationManager:
    def __init__(self,config_filepath=CONFIG_FILE_PATH,
                 schema_filepath=SCHEMA_FILE_PATH,
                 params_filepath=PARAMS_FILE_PATH):
        
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config.model_trainer.root_dir])

    def get_model_evaluation_config(self)->ModelEvaluationConfig:
        config = self.config.model_evaluation
        params = self.params.ElasticNet
        schema = self.schema.TARGET_COLUMN

        create_directories([config.root_dir])

        model_evaluation_config = ModelEvaluationConfig(
            root_dir=config.root_dir,
            test_data_path=config.test_data_path,
            model_path=config.model_path,
            all_params=params,
            metrics_file_name=config.metrics_file_name,
            target_column=next(iter(schema)),
            mlflow_uri = "https://dagshub.com/jkbanjarey/MLOPs_Projects.mlflow"
        )

        return model_evaluation_config


Components

In [52]:
import os
import pandas as pd
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from urllib.parse import urlparse
import mlflow
import mlflow.sklearn
import numpy as np
import joblib

In [53]:
class ModelEvaluation:
    def __init__(self, config:ModelEvaluationConfig):
        self.config = config

    def eval_metrics(self, actual, pred):
        rmse = np.sqrt(mean_squared_error(actual, pred))
        mae = mean_absolute_error(actual, pred)
        r2 = r2_score(actual, pred)
        return rmse, mae, r2

    def log_into_mlflow(self):

        test_data = pd.read_csv(self.config.test_data_path)
        model = joblib.load(self.config.model_path)

        test_x = test_data.drop(columns=[self.config.target_column])
        test_y = test_data[self.config.target_column]

        mlflow.set_tracking_uri(self.config.mlflow_uri)
        tracking_url_type_store = urlparse(mlflow.get_tracking_uri()).scheme

        with mlflow.start_run():
            
            predicted_qualities = model.predict(test_x)

            (rmse, mae, r2) = self.eval_metrics(test_y, predicted_qualities)

            scores = {"rmse": rmse, "mae": mae, "r2": r2}
            save_json(path = Path(self.config.metrics_file_name), data=scores)

            mlflow.log_params(self.config.all_params)

            mlflow.log_metrics(scores)

            if tracking_url_type_store != "file":

                mlflow.sklearn.log_model(model, "model", registered_model_name="ElasticnetModel")

            else:
                mlflow.sklearn.log_model(model,"model")

In [54]:
try:
    config = ConfigurationManager()
    model_evaluation_config = config.get_model_evaluation_config()
    model_evaluation = ModelEvaluation(config=model_evaluation_config)
    model_evaluation.log_into_mlflow()

except Exception as e:
    raise e

[2026-08-29 22:45:53,435: INFO: common: Directory created: artifacts/model_trainer]
[2026-08-29 22:45:53,439: INFO: common: Directory created: artifacts/model_evaluation]
[2026-08-29 22:45:55,443: INFO: common: JSON file saved: artifacts\model_evaluation\metrics.json]


2026/08/29 22:45:56 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
Successfully registered model 'ElasticnetModel'.
2026/08/29 22:46:13 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: ElasticnetModel, version 1
Created version '1' of model 'ElasticnetModel'.


🏃 View run omniscient-mole-724 at: https://dagshub.com/jkbanjarey/MLOPs_Projects.mlflow/#/experiments/0/runs/2aff631f3cd349f0b7551dba7287018f
🧪 View experiment at: https://dagshub.com/jkbanjarey/MLOPs_Projects.mlflow/#/experiments/0
